<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Frequency Domain — Implementation</b></h1>
</div>

## Setup — Environment and Configuration

In [ ]:
# Import the numerical, plotting and image I/O tools required by the FFT workflow.
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

np.set_printoptions(precision=3, suppress=True)

print("NumPy:", np.__version__)
print("Setup: PASS")

### 0.1 Locate the Lab Automatically


In [ ]:
def find_lab_root():
    """Locate the current Frequency-Domain Filtering lab.
    
    Returns
    -------
    Path
        Lab root containing the expected Fourier data and implementation notebook.
    
    Notes
    -----
    The upward search avoids absolute paths while the content checks prevent
    accidentally selecting another project directory.
    """
    cwd = Path.cwd().resolve()

    # Accept execution from either the lab root or its notebooks directory.
    for candidate in [cwd, *cwd.parents]:
        # Require this lab-specific data subtree and implementation notebook before accepting a root.
        if (
            (candidate / "data" / "Fourier").is_dir()
            and (candidate / "notebooks" / "main.ipynb").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the frequency-domain lab root."
    )


LAB_ROOT = find_lab_root()
DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Lab root:", LAB_ROOT)
print("Data dir:", DATA_DIR)
print("Output  :", OUTPUT_DIR)

### 0.2 Reusable helpers

In [ ]:
def load_gray(path):
    """Load one image as float32 grayscale for spectral processing.
    
    Parameters
    ----------
    path : path-like
        Image file to read.
    
    Returns
    -------
    ndarray
        2-D float32 intensity image.
    
    Notes
    -----
    Float storage avoids unsigned overflow and preserves signed/complex-derived
    intermediate values during reconstruction.
    """
    return np.asarray(
        Image.open(path).convert("L"),
        dtype=np.float32
    )


def normalize01(array):
    """Normalize an array to [0,1] for display only.
    
    Parameters
    ----------
    array : array-like
        Numeric array.
    
    Returns
    -------
    ndarray
        float64 array in [0,1].
    
    Notes
    -----
    A constant array is mapped safely to zero. This helper is not used as a
    scientific measurement transform.
    """
    array = np.asarray(array, dtype=np.float64)
    lo = array.min()
    hi = array.max()

    # A constant array has no display dynamic range; map it safely to zero.
    if np.isclose(lo, hi):
        return np.zeros_like(array)

    return (array - lo) / (hi - lo)


def show_gray(ax, image, title, cmap="gray"):
    """Display one scalar image consistently.
    
    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Target axes.
    image : ndarray
        Scalar image to display.
    title : str
        Figure title.
    cmap : str
        Matplotlib colormap.
    
    Returns
    -------
    None
    """
    ax.imshow(image, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")


def save_figure(fig, filename):
    """Save one diagnostic figure under the lab output contract.
    
    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure to persist.
    filename : str
        Output filename relative to outputs/figures/.
    
    Returns
    -------
    None
    """
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path.name)


def fft2_centered(image):
    """Compute the centered 2-D discrete Fourier transform.
    
    Parameters
    ----------
    image : ndarray
        Spatial-domain image.
    
    Returns
    -------
    ndarray
        Complex spectrum with DC shifted to the array center.
    
    Notes
    -----
    Centering changes only index layout; it makes radial distance masks intuitive.
    """
    return np.fft.fftshift(np.fft.fft2(image))


def inverse_fft2_centered(F_shifted):
    """Reconstruct a real image from a centered complex spectrum.
    
    Parameters
    ----------
    F_shifted : ndarray
        Centered complex spectrum.
    
    Returns
    -------
    ndarray
        Real-valued spatial reconstruction.
    
    Notes
    -----
    ifftshift restores NumPy's native FFT ordering before inverse transformation.
    The real part is taken because valid edits preserve conjugate symmetry.
    """
    return np.real(
        np.fft.ifft2(
            np.fft.ifftshift(F_shifted)
        )
    )


def log_magnitude(F_shifted):
    """Compress spectrum magnitude for visualization.
    
    Parameters
    ----------
    F_shifted : ndarray
        Complex centered spectrum.
    
    Returns
    -------
    ndarray
        log(1 + |F|) display image.
    
    Notes
    -----
    Log compression reveals weak coefficients that would be hidden by the large
    dynamic range of raw Fourier magnitudes.
    """
    return np.log1p(np.abs(F_shifted))


def fft_roundtrip_tolerance(image, safety_factor=32.0):
    """Compute a scale-aware numerical FFT reconstruction tolerance.
    
    Parameters
    ----------
    image : ndarray
        Signal being round-tripped.
    safety_factor : float
        Multiplier applied to machine epsilon and signal scale.
    
    Returns
    -------
    float
        Maximum accepted floating-point reconstruction error.
    
    Notes
    -----
    The default factor 32 is deliberately conservative for accumulated FFT
    round-off while remaining far below a meaningful pixel-level error.
    """
    image = np.asarray(image)

    # Preserve the source floating precision when possible; integer input is promoted safely.
    if np.issubdtype(image.dtype, np.floating):
        dtype = image.dtype
    else:
        dtype = np.float64

    # Scale round-trip tolerance to machine precision and signal magnitude.
    eps = np.finfo(dtype).eps
    scale = max(
        1.0,
        float(np.max(np.abs(image)))
    )

    return safety_factor * eps * scale


def mse(reference, test):
    """Compute mean squared error in float64.
    
    Parameters
    ----------
    reference, test : ndarray
        Same-shaped images.
    
    Returns
    -------
    float
        Mean squared difference.
    """
    reference = np.asarray(
        reference,
        dtype=np.float64
    )
    test = np.asarray(
        test,
        dtype=np.float64
    )

    return np.mean(
        (reference - test) ** 2
    )


def psnr(reference, test, peak=255.0):
    """Compute PSNR from MSE.
    
    Parameters
    ----------
    reference, test : ndarray
        Same-shaped images.
    peak : float
        Maximum signal value, 255 by default.
    
    Returns
    -------
    float
        PSNR in decibels, or infinity for zero MSE.
    """
    error = mse(reference, test)

    # Exact reconstruction has zero MSE, which corresponds to infinite PSNR.
    if np.isclose(error, 0.0):
        return np.inf

    return 10 * np.log10(
        (peak ** 2) / error
    )


def mean_gradient_magnitude(image):
    """Summarize retained spatial detail by gradient strength.
    
    Parameters
    ----------
    image : ndarray
        Scalar image.
    
    Returns
    -------
    float
        Mean Euclidean gradient magnitude.
    
    Notes
    -----
    This is a secondary sharpness/detail proxy, not a perceptual quality metric.
    """
    gy, gx = np.gradient(
        np.asarray(
            image,
            dtype=np.float64
        )
    )

    return np.mean(
        np.hypot(gx, gy)
    )


def ringing_overshoot(image, low=0.0, high=255.0):
    """Measure intensity excursions outside an expected range.
    
    Parameters
    ----------
    image : ndarray
        Filtered image.
    low, high : float
        Expected valid intensity limits.
    
    Returns
    -------
    float
        Largest overshoot or undershoot magnitude.
    
    Notes
    -----
    On the synthetic square this provides a simple quantitative proxy for Gibbs
    ringing.
    """
    image = np.asarray(
        image,
        dtype=np.float64
    )

    above = max(
        float(image.max() - high),
        0.0
    )
    below = max(
        float(low - image.min()),
        0.0
    )

    return max(above, below)

## 1. Spatial-Frequency Characterization


In [ ]:
# Contrast low and high spatial frequencies using controlled periodic signals.
width = 512
x = np.linspace(0, 1, width, endpoint=False)

low_signal = np.sin(2 * np.pi * 4 * x)
high_signal = np.sin(2 * np.pi * 32 * x)

low_image = np.tile(low_signal, (180, 1))
high_image = np.tile(high_signal, (180, 1))

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0, 0].plot(x, low_signal)
axes[0, 0].set_title("4 cycles — low frequency")

show_gray(
    axes[0, 1],
    low_image,
    "Low spatial frequency"
)

axes[1, 0].plot(x, high_signal)
axes[1, 0].set_title("32 cycles — high frequency")

show_gray(
    axes[1, 1],
    high_image,
    "High spatial frequency"
)

fig.tight_layout()
save_figure(fig, "01_spatial_frequency.png")
plt.show()


> **Output comment.** The synthetic patterns confirm the meaning of spatial frequency directly: a small number of cycles produces slowly varying intensity, while many cycles produce rapid spatial variation. Brightness amplitude and spatial frequency are separate properties, so a high-frequency pattern is not necessarily brighter than a low-frequency one.


## 2. 1-D DFT Validation with Synthetic Sinusoids


In [ ]:
n = np.arange(128)

# Use known sinusoids so the expected FFT peaks are unambiguous.
signal = (
    np.sin(2 * np.pi * 5 * n / len(n))
    + 0.45 * np.sin(2 * np.pi * 18 * n / len(n))
)

F_signal = np.fft.fft(signal)
frequencies = np.fft.fftfreq(len(signal))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(n, signal)
axes[0].set_title("Signal = two sinusoids")

axes[1].stem(
    frequencies[:64],
    np.abs(F_signal[:64])
)
axes[1].set_title("FFT magnitude")
axes[1].set_xlabel("Frequency")

fig.tight_layout()
save_figure(fig, "02_fft_1d.png")
plt.show()


> **Output comment.** The FFT exposes the two sinusoidal components as distinct spectral peaks at their corresponding frequencies. This validates the transform convention before moving to 2-D images and demonstrates that periodic spatial structure becomes localized in the frequency representation.


## 3. The 2-D Fourier Transform for Images


In [ ]:
# Use one reference image to anchor spectrum, reconstruction and filtering experiments.
house = load_gray(
    DATA_DIR / "Fourier" / "house.png"
)

F_house = fft2_centered(house)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

show_gray(
    axes[0],
    house,
    "House — spatial domain"
)
show_gray(
    axes[1],
    log_magnitude(F_house),
    "Log magnitude spectrum"
)
show_gray(
    axes[2],
    np.angle(F_house),
    "Phase spectrum",
    cmap="twilight"
)

fig.tight_layout()
save_figure(fig, "03_house_spectrum.png")
plt.show()


> **Output comment.** The centered magnitude spectrum concentrates the strongest low-frequency energy near the origin, while finer structures contribute energy farther from the center. The phase image is not visually intuitive by itself, but it retains essential spatial-position information needed for reconstruction.


## 4. 2-D Spectrum Interpretation


In [ ]:
# Build oriented sinusoids to link spatial orientation with spectral structure.
size = 256
coords = np.arange(size)

vertical = np.tile(
    np.sin(2 * np.pi * 16 * coords / size),
    (size, 1)
)

horizontal = vertical.T

xx, yy = np.meshgrid(coords, coords)

diagonal = np.sin(
    2 * np.pi * 12 * (xx + yy) / size
)

patterns = [
    ("Vertical stripes", vertical),
    ("Horizontal stripes", horizontal),
    ("Diagonal stripes", diagonal),
]

fig, axes = plt.subplots(3, 2, figsize=(10, 13))

for row, (name, pattern) in enumerate(patterns):
    F = fft2_centered(pattern)

    show_gray(
        axes[row, 0],
        pattern,
        name
    )
    show_gray(
        axes[row, 1],
        log_magnitude(F),
        f"{name} — spectrum"
    )

fig.tight_layout()
save_figure(fig, "04_orientation_spectra.png")
plt.show()

In [ ]:
# Extend spectrum interpretation from synthetic patterns to real textured images.
fourier_files = [
    "squares.png",
    "textures.jpg",
    "tiled.png",
    "zebra-wall.png",
]

fig, axes = plt.subplots(
    len(fourier_files),
    2,
    figsize=(12, 4 * len(fourier_files))
)

for row, filename in enumerate(fourier_files):
    image = load_gray(
        DATA_DIR / "Fourier" / filename
    )

    F = fft2_centered(image)

    show_gray(
        axes[row, 0],
        image,
        filename
    )
    show_gray(
        axes[row, 1],
        log_magnitude(F),
        f"{filename} — spectrum"
    )

fig.tight_layout()
save_figure(fig, "05_dataset_spectra.png")
plt.show()


> **Output comment.** The orientation experiments show the expected orthogonality between spatial stripes and their spectral peaks: variation in one spatial direction produces frequency energy along the corresponding Fourier axis. Repetitive textures generate structured peak patterns, making the spectrum a useful diagnostic for periodicity and orientation.


## 5. Inverse FFT and Reconstruction


In [ ]:
# Validate the centered FFT/IFFT pipeline before modifying any coefficients.
reconstructed_house = inverse_fft2_centered(
    F_house
)

absolute_error = np.abs(
    house.astype(np.float64)
    - reconstructed_house
)

reconstruction_tolerance = fft_roundtrip_tolerance(
    house
)

print(
    "Maximum reconstruction error:",
    f"{absolute_error.max():.6e}"
)

print(
    "Float-aware tolerance:",
    f"{reconstruction_tolerance:.6e}"
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

show_gray(
    axes[0],
    house,
    "Original"
)
show_gray(
    axes[1],
    reconstructed_house,
    "IFFT reconstruction"
)
show_gray(
    axes[2],
    absolute_error,
    "Absolute error"
)

fig.tight_layout()
save_figure(fig, "06_reconstruction.png")
plt.show()


> **Output comment.** The inverse transform reconstructs the original image to numerical precision when no frequency coefficients are modified. Any residual difference is floating-point round-off, so this round-trip test validates the FFT/shift/IFFT pipeline before filtering is introduced.


## 6. Magnitude–Phase Analysis


In [ ]:
cat = load_gray(
    DATA_DIR / "PhaseMag" / "cat.jpg"
)

wolf = load_gray(
    DATA_DIR / "PhaseMag" / "wolf.jpg"
)

common_size = (256, 256)

cat_r = np.asarray(
    Image.fromarray(
        cat.astype(np.uint8)
    ).resize(common_size),
    dtype=np.float32
)

wolf_r = np.asarray(
    Image.fromarray(
        wolf.astype(np.uint8)
    ).resize(common_size),
    dtype=np.float32
)

# Keep magnitude and phase separate to isolate their roles in reconstruction.
F_cat = np.fft.fft2(cat_r)
F_wolf = np.fft.fft2(wolf_r)

mag_cat = np.abs(F_cat)
phase_cat = np.angle(F_cat)

mag_wolf = np.abs(F_wolf)
phase_wolf = np.angle(F_wolf)

cat_mag_wolf_phase = np.real(
    np.fft.ifft2(
        mag_cat
        * np.exp(1j * phase_wolf)
    )
)

wolf_mag_cat_phase = np.real(
    np.fft.ifft2(
        mag_wolf
        * np.exp(1j * phase_cat)
    )
)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

show_gray(
    axes[0, 0],
    cat_r,
    "Original cat"
)
show_gray(
    axes[0, 1],
    wolf_r,
    "Original wolf"
)
show_gray(
    axes[1, 0],
    cat_mag_wolf_phase,
    "Cat magnitude + wolf phase"
)
show_gray(
    axes[1, 1],
    wolf_mag_cat_phase,
    "Wolf magnitude + cat phase"
)

fig.tight_layout()
save_figure(fig, "07_phase_magnitude_swap.png")
plt.show()

In [ ]:
# Reconstruct with isolated magnitude or phase to compare their structural roles.
magnitude_only = np.real(
    np.fft.ifft2(
        mag_cat
        * np.exp(
            1j * np.zeros_like(phase_cat)
        )
    )
)

phase_only = np.real(
    np.fft.ifft2(
        np.ones_like(mag_cat)
        * np.exp(1j * phase_cat)
    )
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

show_gray(
    axes[0],
    cat_r,
    "Original"
)
show_gray(
    axes[1],
    magnitude_only,
    "Magnitude only"
)
show_gray(
    axes[2],
    phase_only,
    "Phase only"
)

fig.tight_layout()
save_figure(
    fig,
    "08_phase_only_magnitude_only.png"
)
plt.show()


> **Output comment.** The magnitude/phase exchange demonstrates that phase carries most of the recognizable spatial organization, whereas magnitude controls the strength of frequency components. Reconstructions using the phase of one image tend to preserve its structural layout even when paired with the magnitude of another.


## 7. Frequency-Domain Filtering


In [ ]:
def apply_frequency_filter(image, H):
    """Apply a same-size transfer function in the Fourier domain.
    
    Parameters
    ----------
    image : ndarray
        Spatial-domain image.
    H : ndarray
        Real transfer function aligned with the centered spectrum.
    
    Returns
    -------
    result : ndarray
        Reconstructed spatial image.
    F_shifted : ndarray
        Original centered spectrum.
    G_shifted : ndarray
        Filtered centered spectrum.
    
    Notes
    -----
    Shape equality is enforced because each transfer coefficient must map to one
    specific frequency bin.
    """
    image = np.asarray(image, dtype=np.float64)
    H = np.asarray(H, dtype=np.float64)

    # Prevent frequency-bin misalignment before spectral multiplication.
    if image.shape != H.shape:
        raise ValueError(
            f"Image shape {image.shape} and filter shape {H.shape} must match."
        )

    F_shifted = fft2_centered(image)
    # Apply the transfer function in the centered complex spectrum.
    G_shifted = F_shifted * H
    result = inverse_fft2_centered(G_shifted)

    return result, F_shifted, G_shifted


identity_filter = np.ones(
    house.shape,
    dtype=np.float64
)

identity_result, identity_spectrum, identity_filtered_spectrum = (
    apply_frequency_filter(
        house,
        identity_filter
    )
)

identity_error = np.max(
    np.abs(
        identity_result
        - house.astype(np.float64)
    )
)

identity_tolerance = fft_roundtrip_tolerance(
    house
)

assert identity_result.shape == house.shape
assert np.isfinite(identity_result).all()
assert identity_error <= identity_tolerance

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

show_gray(axes[0], house, "Input image")
show_gray(
    axes[1],
    log_magnitude(identity_spectrum),
    "Centered spectrum"
)
show_gray(
    axes[2],
    identity_result,
    f"Identity-filter reconstruction\nmax error={identity_error:.2e}"
)

fig.tight_layout()
save_figure(fig, "07_frequency_filter_pipeline.png")
plt.show()

print(
    "Frequency-domain pipeline: PASS | "
    f"maximum identity error = {identity_error:.3e} | "
    f"tolerance = {identity_tolerance:.3e}"
)


## 8. Frequency Distance Grid


In [ ]:
def frequency_distance_grid(shape):
    """Build radial frequency-bin distance from centered DC.
    
    Parameters
    ----------
    shape : tuple[int, int]
        Spectrum height and width.
    
    Returns
    -------
    ndarray
        Euclidean distance of every bin from the centered origin.
    """
    rows, cols = shape
    # Radial filters are centered on the shifted DC component.
    cy = rows // 2
    cx = cols // 2

    y, x = np.ogrid[:rows, :cols]

    return np.sqrt(
        (y - cy) ** 2
        + (x - cx) ** 2
    )


D_house = frequency_distance_grid(
    house.shape
)

plt.figure(figsize=(6, 5))
plt.imshow(
    D_house,
    cmap="viridis"
)
plt.title("Distance from frequency origin")
plt.colorbar(
    label="Frequency-bin distance"
)
plt.axis("off")
plt.show()

## 9. Ideal, Gaussian, and Butterworth Low-Pass Filters


In [ ]:
# Define the three low-pass families on the same radial frequency grid.
def ideal_low_pass(shape, cutoff):
    """Construct an ideal radial low-pass filter.
    
    Parameters
    ----------
    shape : tuple[int, int]
        Spectrum dimensions.
    cutoff : float
        Radius retained around DC, in frequency bins.
    
    Returns
    -------
    ndarray
        Binary transfer function.
    
    Notes
    -----
    The abrupt transition gives maximum selectivity but promotes spatial ringing.
    """
    D = frequency_distance_grid(shape)
    return (D <= cutoff).astype(np.float32)


def gaussian_low_pass(shape, cutoff):
    """Construct a Gaussian radial low-pass filter.
    
    Parameters
    ----------
    shape : tuple[int, int]
        Spectrum dimensions.
    cutoff : float
        Gaussian radial scale in frequency bins.
    
    Returns
    -------
    ndarray
        Smooth transfer function in (0,1].
    
    Notes
    -----
    The smooth transition reduces ringing at the cost of a less abrupt cutoff.
    """
    D = frequency_distance_grid(shape)

    return np.exp(
        -(D ** 2)
        / (2 * cutoff ** 2)
    )


def butterworth_low_pass(
    shape,
    cutoff,
    order=2
):
    """Construct a Butterworth radial low-pass filter.
    
    Parameters
    ----------
    shape : tuple[int, int]
        Spectrum dimensions.
    cutoff : float
        Nominal cutoff radius.
    order : int
        Positive order controlling transition steepness.
    
    Returns
    -------
    ndarray
        Transfer function in (0,1].
    
    Notes
    -----
    Order provides a controlled compromise: increasing it approaches the ideal
    filter and therefore also increases ringing risk.
    """
    D = frequency_distance_grid(shape)

    return 1.0 / (
        1.0
        + (
            D
            / max(float(cutoff), 1e-12)
        ) ** (2 * order)
    )


# Use one shared moderate cutoff so filter-family differences dominate the comparison.
cutoff = 30

H_ideal_lp = ideal_low_pass(
    house.shape,
    cutoff
)

H_gaussian_lp = gaussian_low_pass(
    house.shape,
    cutoff
)

H_butterworth_lp = butterworth_low_pass(
    house.shape,
    cutoff,
    order=2
)

house_ideal_lp, _, _ = apply_frequency_filter(
    house,
    H_ideal_lp
)

house_gaussian_lp, _, _ = apply_frequency_filter(
    house,
    H_gaussian_lp
)

house_butterworth_lp, _, _ = apply_frequency_filter(
    house,
    H_butterworth_lp
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_gray(
    axes[0, 0],
    H_ideal_lp,
    "Ideal LPF"
)

show_gray(
    axes[0, 1],
    H_gaussian_lp,
    "Gaussian LPF"
)

show_gray(
    axes[0, 2],
    H_butterworth_lp,
    "Butterworth LPF"
)

show_gray(
    axes[1, 0],
    house_ideal_lp,
    "Ideal result"
)

show_gray(
    axes[1, 1],
    house_gaussian_lp,
    "Gaussian result"
)

show_gray(
    axes[1, 2],
    house_butterworth_lp,
    "Butterworth result"
)

fig.tight_layout()
save_figure(
    fig,
    "09_lpf_comparison.png"
)
plt.show()


lpf_quality_metrics = {
    "Ideal": {
        "mse": mse(
            house,
            house_ideal_lp
        ),
        "psnr": psnr(
            house,
            house_ideal_lp
        ),
        "edge_strength": mean_gradient_magnitude(
            house_ideal_lp
        ),
    },
    "Gaussian": {
        "mse": mse(
            house,
            house_gaussian_lp
        ),
        "psnr": psnr(
            house,
            house_gaussian_lp
        ),
        "edge_strength": mean_gradient_magnitude(
            house_gaussian_lp
        ),
    },
    "Butterworth n=2": {
        "mse": mse(
            house,
            house_butterworth_lp
        ),
        "psnr": psnr(
            house,
            house_butterworth_lp
        ),
        "edge_strength": mean_gradient_magnitude(
            house_butterworth_lp
        ),
    },
}

In [ ]:
# Sweep orders 1→8 at fixed cutoff so only transition steepness changes.
orders = [1, 2, 4, 8]
butterworth_order_results = []

fig, axes = plt.subplots(
    len(orders),
    2,
    figsize=(11, 15)
)

for row, order in enumerate(orders):
    H = butterworth_low_pass(
        house.shape,
        cutoff,
        order=order
    )

    result, _, _ = apply_frequency_filter(
        house,
        H
    )

    butterworth_order_results.append({
        "order": order,
        "mse": mse(
            house,
            result
        ),
        "edge_strength": mean_gradient_magnitude(
            result
        ),
    })

    show_gray(
        axes[row, 0],
        H,
        f"Butterworth n={order}"
    )

    show_gray(
        axes[row, 1],
        result,
        f"Result n={order}"
    )

fig.tight_layout()
save_figure(
    fig,
    "10_butterworth_orders.png"
)
plt.show()


> **Output comment.** All three masks suppress high-frequency content, but their transition profiles differ. The ideal filter performs the sharpest frequency separation, Gaussian filtering changes smoothly with distance from the origin, and Butterworth behavior lies between them with controllable steepness. These spectral differences explain the different spatial artifacts observed later.


## 10. Ringing and the Gibbs Phenomenon


In [ ]:
# Use a sharp synthetic edge so ringing from abrupt spectral truncation is measurable.
square = np.zeros(
    (256, 256),
    dtype=np.float32
)

square[64:192, 64:192] = 255.0

# Use a relatively sharp low cutoff so ringing is obvious on the synthetic square.
ring_cutoff = 22

Hi = ideal_low_pass(
    square.shape,
    ring_cutoff
)

Hg = gaussian_low_pass(
    square.shape,
    ring_cutoff
)

Hb = butterworth_low_pass(
    square.shape,
    ring_cutoff,
    order=2
)

square_i, _, _ = apply_frequency_filter(
    square,
    Hi
)

square_g, _, _ = apply_frequency_filter(
    square,
    Hg
)

square_b, _, _ = apply_frequency_filter(
    square,
    Hb
)

center_row = square.shape[0] // 2

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 10)
)

show_gray(
    axes[0, 0],
    square,
    "Original square"
)

show_gray(
    axes[0, 1],
    square_i,
    "Ideal LPF — ringing"
)

axes[1, 0].plot(
    square[center_row],
    label="Original"
)

axes[1, 0].plot(
    square_i[center_row],
    label="Ideal"
)

axes[1, 0].plot(
    square_g[center_row],
    label="Gaussian"
)

axes[1, 0].plot(
    square_b[center_row],
    label="Butterworth"
)

axes[1, 0].set_title(
    "Intensity profile through edge"
)
axes[1, 0].legend()

show_gray(
    axes[1, 1],
    np.abs(square_i - square),
    "Ideal LPF difference"
)

fig.tight_layout()
save_figure(
    fig,
    "11_ringing.png"
)
plt.show()


ringing_by_method = {
    "Ideal": ringing_overshoot(
        square_i
    ),
    "Gaussian": ringing_overshoot(
        square_g
    ),
    "Butterworth n=2": ringing_overshoot(
        square_b
    ),
}

butterworth_order_ringing = []

for order in orders:
    H_order = butterworth_low_pass(
        square.shape,
        ring_cutoff,
        order=order
    )

    square_order, _, _ = apply_frequency_filter(
        square,
        H_order
    )

    butterworth_order_ringing.append({
        "order": order,
        "ringing": ringing_overshoot(
            square_order
        ),
    })


> **Output comment.** The ideal filter's abrupt cutoff produces oscillatory sidelobes in the spatial domain, visible as ringing near strong edges. Smoother Gaussian and lower-order Butterworth transitions reduce this effect, illustrating the trade-off between sharp spectral selectivity and spatial-domain artifacts.


## 11. High-Pass Filtering


In [ ]:
# Derive complementary high-pass filters from the validated low-pass masks.
def high_pass_from_low_pass(H_low):
    """Construct the normalized complementary high-pass response.
    
    Parameters
    ----------
    H_low : ndarray
        Low-pass transfer function bounded in [0,1].
    
    Returns
    -------
    ndarray
        1 - H_low.
    
    Notes
    -----
    Complement construction keeps low- and high-frequency components algebraically
    consistent.
    """
    return 1.0 - H_low


H_ideal_hp = high_pass_from_low_pass(
    H_ideal_lp
)

H_gaussian_hp = high_pass_from_low_pass(
    H_gaussian_lp
)

H_butterworth_hp = high_pass_from_low_pass(
    H_butterworth_lp
)

house_i_hp, _, _ = apply_frequency_filter(
    house,
    H_ideal_hp
)

house_g_hp, _, _ = apply_frequency_filter(
    house,
    H_gaussian_hp
)

house_b_hp, _, _ = apply_frequency_filter(
    house,
    H_butterworth_hp
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_gray(
    axes[0, 0],
    H_ideal_hp,
    "Ideal HPF"
)

show_gray(
    axes[0, 1],
    H_gaussian_hp,
    "Gaussian HPF"
)

show_gray(
    axes[0, 2],
    H_butterworth_hp,
    "Butterworth HPF"
)

show_gray(
    axes[1, 0],
    normalize01(house_i_hp),
    "Ideal HP response"
)

show_gray(
    axes[1, 1],
    normalize01(house_g_hp),
    "Gaussian HP response"
)

show_gray(
    axes[1, 2],
    normalize01(house_b_hp),
    "Butterworth HP response"
)

fig.tight_layout()
save_figure(
    fig,
    "12_high_pass.png"
)
plt.show()


> **Output comment.** High-pass filtering suppresses slowly varying content and emphasizes edges and fine detail. The result is not a normal enhanced image by itself; it is primarily a detail component and can also amplify high-frequency noise.


## 12. High-Boost Sharpening


In [ ]:
# Reinject controlled high-frequency detail to study sharpening gain.
high_detail = house_g_hp

# Sweep from mild to aggressive detail reinjection to expose sharpening artifacts.
high_boost_gains = [0.5, 1.0, 1.5, 2.0]
high_boost_results = []

fig, axes = plt.subplots(
    1,
    len(high_boost_gains) + 1,
    figsize=(4 * (len(high_boost_gains) + 1), 5)
)

show_gray(
    axes[0],
    house,
    "Original"
)

for col, gain in enumerate(
    high_boost_gains,
    start=1
):
    boosted_unclipped = (
        house.astype(np.float64)
        + gain * high_detail
    )

    clipped_mask = (
        (boosted_unclipped < 0.0)
        | (boosted_unclipped > 255.0)
    )

    boosted = np.clip(
        boosted_unclipped,
        0.0,
        255.0
    )

    clipping_percent = (
        100.0 * np.mean(clipped_mask)
    )

    high_boost_results.append({
        "gain": gain,
        "clipping_percent": clipping_percent,
        "edge_strength": mean_gradient_magnitude(
            boosted
        ),
    })

    show_gray(
        axes[col],
        boosted,
        (
            f"k={gain:.1f}\n"
            f"clipped={clipping_percent:.2f}%"
        )
    )

fig.tight_layout()
save_figure(
    fig,
    "13_high_boost.png"
)
plt.show()

# Choose a moderate final high-boost gain after inspecting the broader gain sweep.
k = 1.2
high_boost = np.clip(
    house.astype(np.float64)
    + k * high_detail,
    0.0,
    255.0
)


> **Output comment.** Adding a scaled high-frequency component back to the original image increases local edge contrast while preserving the low-frequency scene content. The gain controls enhancement strength, but large gains increase clipping, halos, and noise sensitivity.


## 13. Convolution Theorem


In [ ]:
conv_image = house[:96, :96].astype(np.float64)

conv_kernel = np.ones((5, 5), dtype=np.float64)
conv_kernel /= conv_kernel.sum()

pad_y = conv_kernel.shape[0] // 2
pad_x = conv_kernel.shape[1] // 2

spatial_padded = np.pad(
    conv_image,
    ((pad_y, pad_y), (pad_x, pad_x)),
    mode="constant"
)

windows = np.lib.stride_tricks.sliding_window_view(
    spatial_padded,
    conv_kernel.shape
)

spatial_conv = np.sum(
    windows * conv_kernel[::-1, ::-1],
    axis=(-2, -1)
)

# Zero-pad to obtain linear, not circular, convolution.
fft_shape = (
    conv_image.shape[0] + conv_kernel.shape[0] - 1,
    conv_image.shape[1] + conv_kernel.shape[1] - 1,
)

fft_full = np.real(
    np.fft.ifft2(
        np.fft.fft2(conv_image, s=fft_shape)
        * np.fft.fft2(conv_kernel, s=fft_shape)
    )
)

fft_conv = fft_full[
    pad_y:pad_y + conv_image.shape[0],
    pad_x:pad_x + conv_image.shape[1]
]

conv_error = np.abs(spatial_conv - fft_conv)

assert spatial_conv.shape == fft_conv.shape
assert np.isfinite(fft_conv).all()
assert conv_error.max() < 1e-8

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

show_gray(
    axes[0],
    spatial_conv,
    "Direct spatial convolution"
)
show_gray(
    axes[1],
    fft_conv,
    "Zero-padded FFT convolution"
)
show_gray(
    axes[2],
    conv_error,
    f"Absolute difference\nmax={conv_error.max():.2e}"
)

fig.tight_layout()
save_figure(fig, "13_convolution_validation.png")
plt.show()

print(
    "Convolution validation: PASS | "
    f"maximum absolute difference = {conv_error.max():.3e}"
)


> **Output comment.** The numerical comparison confirms that properly padded spatial convolution and frequency-domain multiplication represent the same linear filtering operation. Any mismatch near boundaries points to padding or circular-convolution assumptions rather than a failure of the theorem.


## 14. Band-Pass and Band-Reject Filters


In [ ]:
# Restrict energy to a chosen annulus to isolate intermediate spatial frequencies.
def ideal_band_pass(
    shape,
    low_cutoff,
    high_cutoff
):
    """Construct a binary annular band-pass filter.
    
    Parameters
    ----------
    shape : tuple[int, int]
        Spectrum dimensions.
    low_cutoff, high_cutoff : float
        Inner and outer retained radii.
    
    Returns
    -------
    ndarray
        Binary transfer function retaining only the selected annulus.
    """
    D = frequency_distance_grid(shape)

    return (
        (D >= low_cutoff)
        & (D <= high_cutoff)
    ).astype(np.float32)


H_band = ideal_band_pass(
    house.shape,
    15,
    55
)

H_band_reject = 1.0 - H_band

house_band, _, _ = apply_frequency_filter(
    house,
    H_band
)

house_band_reject, _, _ = apply_frequency_filter(
    house,
    H_band_reject
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 10)
)

show_gray(
    axes[0, 0],
    H_band,
    "Band-pass mask"
)

show_gray(
    axes[0, 1],
    normalize01(house_band),
    "Band-pass content"
)

show_gray(
    axes[1, 0],
    H_band_reject,
    "Band-reject mask"
)

show_gray(
    axes[1, 1],
    house_band_reject,
    "Band-reject result"
)

fig.tight_layout()
save_figure(
    fig,
    "14_band_filters.png"
)
plt.show()

## 15. Periodic Interference Analysis


In [ ]:
# Compare periodic interference in both image and spectrum before filtering.
interference_files = [
    "astronaut-interference.tif",
    "car-moire-pattern.tif",
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 10)
)

for row, filename in enumerate(interference_files):
    image = load_gray(
        DATA_DIR
        / "Frequency"
        / filename
    )

    F = fft2_centered(image)

    show_gray(
        axes[row, 0],
        image,
        filename
    )

    show_gray(
        axes[row, 1],
        log_magnitude(F),
        f"{filename} — spectrum"
    )

fig.tight_layout()
save_figure(
    fig,
    "15_periodic_noise_spectra.png"
)
plt.show()


> **Output comment.** Periodic interference appears as localized off-center spectral peaks because a repeated spatial pattern is concentrated at specific frequencies. This makes Fourier analysis especially effective when the unwanted disturbance is periodic rather than broadband.


## 16. Spectral Peak Detection


In [ ]:
def strongest_spectral_peaks(
    F_shifted,
    # Inspect several candidates because periodic interference appears in symmetric peak pairs.
number_of_peaks=8,
    center_exclusion_radius=20,
    min_separation=10
):
    """Select separated off-center spectral maxima.
    
    Parameters
    ----------
    F_shifted : ndarray
        Centered complex spectrum.
    number_of_peaks : int
        Maximum candidates returned.
    center_exclusion_radius : float
        Radius around DC ignored during ranking.
    min_separation : float
        Minimum Euclidean spacing between accepted peaks.
    
    Returns
    -------
    list[tuple[int, int]]
        Candidate peak coordinates.
    
    Notes
    -----
    DC exclusion prevents natural low-frequency energy from dominating. Separation
    prevents repeatedly selecting neighboring pixels from the same broad lobe.
    """
    magnitude = log_magnitude(
        F_shifted
    ).copy()

    rows, cols = magnitude.shape
    cy, cx = rows // 2, cols // 2

    yy, xx = np.ogrid[:rows, :cols]

    center_mask = (
        (yy - cy) ** 2
        + (xx - cx) ** 2
        <= center_exclusion_radius ** 2
    )

    # Exclude dominant low frequencies before ranking interference peaks.
    magnitude[center_mask] = -np.inf

    flat_order = np.argsort(
        magnitude.ravel()
    )[::-1]

    selected = []

    for flat_index in flat_order:
        y, x = np.unravel_index(
            flat_index,
            magnitude.shape
        )

        separated = all(
            (y - py) ** 2
            + (x - px) ** 2
            >= min_separation ** 2
            for py, px in selected
        )

        # Accept a peak only when it adds a genuinely distinct spectral location.
        if separated:
            selected.append((y, x))

        # Stop once the requested candidate budget is filled.
        if len(selected) >= number_of_peaks:
            break

    return selected

## 17. Notch-Reject Filtering


In [ ]:
def notch_reject_mask(
    shape,
    peak_locations,
    radius=5
):
    """Build a conjugate-symmetric notch-reject mask.
    
    Parameters
    ----------
    shape : tuple[int, int]
        Spectrum dimensions.
    peak_locations : sequence[tuple[int, int]]
        Interference peak coordinates.
    radius : float
        Radius suppressed around each peak.
    
    Returns
    -------
    ndarray
        Float mask containing ones except at rejected neighborhoods.
    
    Notes
    -----
    Every selected peak is paired with its conjugate-symmetric counterpart so the
    inverse transform remains real-valued. Larger radius improves suppression but
    removes more legitimate neighboring frequencies.
    """
    rows, cols = shape
    cy, cx = rows // 2, cols // 2

    yy, xx = np.ogrid[:rows, :cols]

    mask = np.ones(
        shape,
        dtype=np.float32
    )

    for py, px in peak_locations:
        d1 = (
            (yy - py) ** 2
            + (xx - px) ** 2
        )

        mask[d1 <= radius ** 2] = 0.0

        # Reject each peak with its conjugate-symmetric counterpart.
        sym_y = 2 * cy - py
        sym_x = 2 * cx - px

        d2 = (
            (yy - sym_y) ** 2
            + (xx - sym_x) ** 2
        )

        mask[d2 <= radius ** 2] = 0.0

    return mask

In [ ]:
astronaut = load_gray(
    DATA_DIR
    / "Frequency"
    / "astronaut-interference.tif"
)

F_astronaut = fft2_centered(
    astronaut
)

# Detect several separated off-center peaks; the parameters suppress the DC lobe
# and prevent one broad spectral structure from being selected repeatedly.
astronaut_peaks = strongest_spectral_peaks(
    F_astronaut,
    number_of_peaks=8,
    center_exclusion_radius=25,
    min_separation=12
)

# Test narrow-to-broad notches so interference suppression can be traded against spectral loss.
notch_radii = [2, 4, 6, 8]
notch_radius_results = []

astronaut_energy = np.sum(
    np.abs(F_astronaut) ** 2
)

# Sweep notch radius to quantify suppression versus spectral damage.
for radius in notch_radii:
    H_radius = notch_reject_mask(
        astronaut.shape,
        astronaut_peaks[:4],
        radius=radius
    )

    removed_energy = np.sum(
        np.abs(F_astronaut) ** 2
        * (1.0 - H_radius)
    )

    notch_radius_results.append({
        "radius": radius,
        "removed_energy_percent": (
            100.0
            * removed_energy
            / max(astronaut_energy, 1e-12)
        ),
    })

# Radius 4 is the mid-range compromise used after the sensitivity sweep.
selected_notch_radius = 4

astronaut_notch = notch_reject_mask(
    astronaut.shape,
    astronaut_peaks[:4],
    radius=selected_notch_radius
)

astronaut_filtered = inverse_fft2_centered(
    F_astronaut
    * astronaut_notch
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 10)
)

show_gray(
    axes[0, 0],
    astronaut,
    "Original interference"
)

show_gray(
    axes[0, 1],
    log_magnitude(F_astronaut),
    "Original spectrum"
)

show_gray(
    axes[1, 0],
    astronaut_notch,
    f"Notch mask — r={selected_notch_radius}"
)

show_gray(
    axes[1, 1],
    astronaut_filtered,
    "After notch filtering"
)

fig.tight_layout()
save_figure(
    fig,
    "16_notch_filter.png"
)
plt.show()

print(
    "Candidate peaks:",
    astronaut_peaks
)


> **Output comment.** Notch rejection removes narrow spectral neighborhoods around the detected interference frequencies while preserving most of the remaining spectrum. The conjugate-symmetric notch pairs are essential for maintaining a physically consistent real-valued reconstruction.


## 18. Moiré Removal


In [ ]:
# Detect dominant moiré peaks and suppress only their localized spectral support.
car_moire = load_gray(
    DATA_DIR
    / "Frequency"
    / "car-moire-pattern.tif"
)

F_car = fft2_centered(
    car_moire
)

# Request extra candidates first; a wider DC exclusion reflects this image's broader
# central lobe, and spacing prevents duplicate selections from one spectral cluster.
car_peaks = strongest_spectral_peaks(
    F_car,
    number_of_peaks=10,
    center_exclusion_radius=30,
    min_separation=12
)

# Limit removal to the six strongest candidates and reuse the moderate radius=4
# selected in the previous notch sensitivity study to reduce over-filtering.
car_notch = notch_reject_mask(
    car_moire.shape,
    car_peaks[:6],
    radius=4
)

car_filtered = inverse_fft2_centered(
    F_car * car_notch
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5)
)

show_gray(
    axes[0],
    car_moire,
    "Car with moiré"
)

show_gray(
    axes[1],
    log_magnitude(F_car),
    "Moiré spectrum"
)

show_gray(
    axes[2],
    car_filtered,
    "Notch-filtered result"
)

fig.tight_layout()
save_figure(
    fig,
    "17_moire_removal.png"
)
plt.show()


> **Output comment.** The moiré example shows that structured interference can often be isolated more cleanly in the Fourier domain than in the image itself. Successful suppression depends on removing only the interference peaks; overly broad notches would also erase legitimate periodic texture.


## 19. Low-Frequency Illumination Correction


In [ ]:
spotshade = load_gray(
    DATA_DIR
    / "Frequency"
    / "text-spotshade.tif"
)

illumination_filter = gaussian_low_pass(
    spotshade.shape,
    cutoff=18
)

illumination, _, _ = apply_frequency_filter(
    spotshade,
    illumination_filter
)

epsilon = 1e-6

# Divide by the low-frequency illumination estimate before renormalization.
corrected = spotshade / (
    illumination + epsilon
)

corrected = normalize01(
    corrected
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    spotshade,
    "Original shaded image"
)

show_gray(
    axes[1],
    illumination,
    "Estimated illumination"
)

show_gray(
    axes[2],
    corrected,
    "Normalized image"
)

fig.tight_layout()
save_figure(
    fig,
    "18_shading_correction.png"
)
plt.show()


> **Output comment.** Slow shading variation is concentrated near the spectral origin. Estimating and compensating this low-frequency component reduces uneven illumination while retaining faster reflectance detail. The result supports the assumption that illumination varies more slowly than the scene structure of interest.


## 20. Cutoff Sensitivity


In [ ]:
# Sweep cutoffs from strong smoothing to near-pass-through to expose sensitivity.
# Cover aggressive, moderate, mild and weak Gaussian low-pass regimes.
cutoffs = [10, 25, 60, 120]
cutoff_sensitivity_results = []

fig, axes = plt.subplots(
    2,
    len(cutoffs),
    figsize=(4 * len(cutoffs), 8)
)

for col, current_cutoff in enumerate(cutoffs):
    H = gaussian_low_pass(
        house.shape,
        current_cutoff
    )

    filtered, _, _ = apply_frequency_filter(
        house,
        H
    )

    cutoff_sensitivity_results.append({
        "cutoff": current_cutoff,
        "mse": mse(
            house,
            filtered
        ),
        "edge_strength": mean_gradient_magnitude(
            filtered
        ),
    })

    show_gray(
        axes[0, col],
        H,
        f"Gaussian LPF\nD0={current_cutoff}"
    )

    show_gray(
        axes[1, col],
        filtered,
        f"Result\nD0={current_cutoff}"
    )

fig.tight_layout()
save_figure(
    fig,
    "19_cutoff_sensitivity.png"
)
plt.show()


> **Output comment.** The cutoff sweep makes the filtering trade-off explicit: smaller cutoffs remove more high-frequency information and increase smoothing, while larger cutoffs preserve more detail but perform weaker noise or interference suppression. A cutoff should therefore be selected from the observed trend rather than chosen arbitrarily.


## 21. Quantitative Checks


In [ ]:
# Compare all low-pass families at one fixed cutoff before changing parameters.
print(
    "Low-pass comparison at cutoff D0=30"
)

for name, metrics in lpf_quality_metrics.items():
    print(
        f"{name:20s} "
        f"MSE={metrics['mse']:10.3f} "
        f"PSNR={metrics['psnr']:7.2f} dB "
        f"Edge={metrics['edge_strength']:8.3f}"
    )


## 22. Validation Checks


In [ ]:
validation_reconstruction = inverse_fft2_centered(
    fft2_centered(house)
)

validation_error = np.abs(
    house.astype(np.float64)
    - validation_reconstruction
)

validation_tolerance = fft_roundtrip_tolerance(
    house
)

assert validation_reconstruction.shape == house.shape
assert np.isfinite(validation_reconstruction).all()

assert (
    validation_error.max()
    <= validation_tolerance
), (
    "FFT/IFFT round-trip error exceeded the "
    "dtype-aware tolerance: "
    f"{validation_error.max():.6e} > "
    f"{validation_tolerance:.6e}"
)

# Validate transfer-function bounds and DC behavior, not only output shape.
filters_to_check = [
    H_ideal_lp,
    H_gaussian_lp,
    H_butterworth_lp,
    H_ideal_hp,
    H_gaussian_hp,
    H_butterworth_hp,
]

for H in filters_to_check:
    assert H.shape == house.shape
    assert np.isfinite(H).all()
    assert H.min() >= 0
    assert H.max() <= 1

center = (
    house.shape[0] // 2,
    house.shape[1] // 2
)

assert np.isclose(
    H_ideal_lp[center],
    1.0
)

assert np.isclose(
    H_gaussian_lp[center],
    1.0
)

assert np.isclose(
    H_butterworth_lp[center],
    1.0
)

assert np.isclose(
    H_ideal_hp[center],
    0.0
)

assert np.isclose(
    H_gaussian_hp[center],
    0.0
)

assert np.isclose(
    H_butterworth_hp[center],
    0.0
)

print(
    "PASS — reconstruction and filter "
    "sanity checks"
)

print(
    "Round-trip max error:",
    f"{validation_error.max():.6e}"
)

print(
    "Accepted tolerance:",
    f"{validation_tolerance:.6e}"
)

## 23. Failure Modes and Diagnostic Signatures


In [ ]:
# Report diagnostics that expose numerical or spectral failure modes.
raw_dynamic_range = (
    np.max(np.abs(F_house))
    / max(
        np.median(np.abs(F_house)),
        1e-12
    )
)

failure_diagnostics = {
    "FFT raw dynamic-range ratio": raw_dynamic_range,
    "FFT/IFFT round-trip max error": validation_error.max(),
    "FFT/IFFT accepted tolerance": validation_tolerance,
    "Convolution max error": conv_error.max(),
    "Ideal LPF ringing overshoot": ringing_by_method["Ideal"],
    "Gaussian LPF ringing overshoot": ringing_by_method["Gaussian"],
    "High-boost max clipping (%)": max(
        row["clipping_percent"]
        for row in high_boost_results
    ),
}

print("Failure-mode diagnostic summary")

for name, value in failure_diagnostics.items():
    print(
        f"  {name:36s}: {value:.6e}"
    )

print()
print(
    "Interpretation: large raw spectral dynamic range "
    "justifies log-magnitude visualization; "
    "round-trip and convolution errors verify numerical "
    "correctness; ringing and clipping quantify the main "
    "artifact risks already observed above."
)


## 24. Parameter Sensitivity and Controlled Experiments


In [ ]:
# Consolidate parameter sweeps so method choices are evidence-based.
print("Controlled parameter sensitivity summary")

print("\nGaussian cutoff sweep")
for row in cutoff_sensitivity_results:
    print(
        f"  D0={row['cutoff']:3d} | "
        f"MSE={row['mse']:10.3f} | "
        f"Edge={row['edge_strength']:8.3f}"
    )

print("\nButterworth order sweep")
ringing_lookup = {
    row["order"]: row["ringing"]
    for row in butterworth_order_ringing
}

for row in butterworth_order_results:
    print(
        f"  n={row['order']:2d} | "
        f"MSE={row['mse']:10.3f} | "
        f"Edge={row['edge_strength']:8.3f} | "
        f"Ringing={ringing_lookup[row['order']]:8.3f}"
    )

print("\nHigh-boost gain sweep")
for row in high_boost_results:
    print(
        f"  k={row['gain']:.1f} | "
        f"Clipped={row['clipping_percent']:7.3f}% | "
        f"Edge={row['edge_strength']:8.3f}"
    )

print("\nNotch-radius sweep")
for row in notch_radius_results:
    print(
        f"  r={row['radius']:2d} | "
        f"Removed spectral energy="
        f"{row['removed_energy_percent']:.6f}%"
    )


## 25. Method Selection and Technical Discussion


In [ ]:
# Compare candidate filters using the same fidelity and edge-preservation metrics.
method_selection_results = []

for method, metrics in lpf_quality_metrics.items():
    method_selection_results.append({
        "method": method,
        "mse": metrics["mse"],
        "psnr": metrics["psnr"],
        "edge_strength": metrics["edge_strength"],
        "ringing": ringing_by_method[method],
    })

print(
    "Method                 "
    "MSE        PSNR(dB)   "
    "EdgeStrength   Ringing"
)

for row in method_selection_results:
    print(
        f"{row['method']:20s} "
        f"{row['mse']:10.3f} "
        f"{row['psnr']:10.2f} "
        f"{row['edge_strength']:14.3f} "
        f"{row['ringing']:9.3f}"
    )

lowest_change = min(
    method_selection_results,
    key=lambda row: row["mse"]
)

lowest_ringing = min(
    method_selection_results,
    key=lambda row: row["ringing"]
)

strongest_edges = max(
    method_selection_results,
    key=lambda row: row["edge_strength"]
)

print()
print(
    "Lowest numerical change :",
    lowest_change["method"]
)
print(
    "Lowest measured ringing :",
    lowest_ringing["method"]
)
print(
    "Highest retained edge strength:",
    strongest_edges["method"]
)


> **Output comment.** The experiments show that filter choice must be tied to the processing objective. Gaussian or low-order Butterworth filters are preferable when ringing must be minimized, sharper filters provide stronger selectivity when artifacts are acceptable, notch filters target localized periodic interference, and high-boost filtering is appropriate for detail enhancement rather than denoising.


## 26. Integrated Frequency-Domain Workflow


In [ ]:
# Package the validated steps into one reusable end-to-end filtering workflow.
def run_frequency_domain_workflow(
    image_path,
    filter_family="butterworth",
    cutoff=30,
    order=2
):
    """Run the validated transform-filter-reconstruct workflow.
    
    Parameters
    ----------
    image_path : path-like
        Input grayscale image.
    filter_family : {"ideal", "gaussian", "butterworth"}
        Low-pass family.
    cutoff : float
        Radial cutoff/scale in frequency bins.
    order : int
        Butterworth order when that family is selected.
    
    Returns
    -------
    dict
        Configuration plus fidelity and transfer-function diagnostics.
    
    Notes
    -----
    Butterworth order 2 is the default because it provides moderate selectivity
    without the strong ringing of an ideal transition.
    """
    image = load_gray(image_path)

    builders = {
        "ideal": lambda: ideal_low_pass(
            image.shape,
            cutoff
        ),
        "gaussian": lambda: gaussian_low_pass(
            image.shape,
            cutoff
        ),
        "butterworth": lambda: butterworth_low_pass(
            image.shape,
            cutoff,
            order=order
        ),
    }

    # Fail explicitly rather than silently falling back to another filter family.
    if filter_family not in builders:
        raise ValueError(
            "filter_family must be one of: "
            + ", ".join(builders)
        )

    H = builders[filter_family]()

    result, F_input, F_filtered = apply_frequency_filter(
        image,
        H
    )

    assert result.shape == image.shape
    assert H.shape == image.shape
    assert np.isfinite(result).all()
    assert np.isfinite(H).all()
    assert H.min() >= 0.0
    assert H.max() <= 1.0

    report = {
        "image": image_path.name,
        "shape": image.shape,
        "filter_family": filter_family,
        "cutoff": cutoff,
        "order": (
            order
            # Report order only when the selected family actually uses that parameter.
            if filter_family == "butterworth"
            else None
        ),
        "mse_vs_input": mse(image, result),
        "psnr_vs_input_db": psnr(image, result),
        "filter_min": float(H.min()),
        "filter_max": float(H.max()),
    }

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))

    show_gray(axes[0], image, "Input")
    show_gray(
        axes[1],
        log_magnitude(F_input),
        "Centered spectrum"
    )
    show_gray(axes[2], H, "Selected filter")
    show_gray(
        axes[3],
        log_magnitude(F_filtered),
        "Filtered spectrum"
    )
    show_gray(
        axes[4],
        result,
        "Reconstructed result"
    )

    fig.tight_layout()
    save_figure(fig, "23_integrated_workflow.png")
    plt.show()

    return report


final_report = run_frequency_domain_workflow(
    DATA_DIR / "Fourier" / "house.png",
    filter_family="butterworth",
    cutoff=30,
    order=2
)

print("Final workflow configuration")
for key, value in final_report.items():
    print(f"  {key:20s}: {value}")


> **Output comment.** The integrated workflow combines transform generation, filter construction, spectral multiplication, inverse reconstruction, and diagnostics into one reproducible path. Its value is consistency: the same coordinate convention, normalization, and validation rules are applied to every experiment rather than being reimplemented ad hoc.


## Final Analysis & Interpretation

### Main findings

- Synthetic patterns verify that spatial periodicity maps to localized Fourier components and that spectrum orientation reflects the direction of spatial variation.
- FFT followed by inverse FFT reconstructs the original image to numerical precision when coefficients are unchanged, validating the transform convention before filtering.
- Phase carries much of the recognizable spatial organization, while magnitude controls the strength of spectral components.
- Ideal, Gaussian, and Butterworth low-pass filters differ mainly in transition smoothness. Sharper spectral transitions improve selectivity but increase spatial ringing.
- High-pass components isolate fine detail and edges; reinjecting them through high-boost filtering increases sharpness but also raises clipping and noise sensitivity.
- Spatial convolution and frequency-domain multiplication agree when boundary and padding assumptions are handled consistently.
- Periodic interference produces localized off-center spectral peaks, making conjugate-symmetric notch rejection effective when those peaks can be isolated without removing legitimate texture.
- Cutoff, Butterworth order, notch radius, and high-boost gain all express measurable trade-offs rather than arbitrary constants.
- The integrated workflow passes the numerical transform, filter-range, reconstruction, and output checks.

### Engineering interpretation

Frequency-domain processing is most powerful when the spatial phenomenon has a meaningful spectral signature. Smooth low-pass transitions trade frequency selectivity for reduced ringing, while notch filters trade interference removal for the risk of deleting nearby legitimate frequencies. The Fourier domain therefore does not eliminate design choices; it makes their spectral consequences explicit.

### Limitations

Periodic-noise peak detection is heuristic and image-dependent. Radial filters assume isotropic frequency behavior, and MSE/PSNR against the original are not always meaningful when the processing objective is intentional enhancement rather than faithful reconstruction.

### Final conclusion

The notebook demonstrates and validates the full link between spatial structure and spectral representation: transform analysis, low/high/band filtering, ringing, sharpening, convolution equivalence, periodic-noise suppression, parameter sensitivity, and an integrated reusable workflow.
